# 5. Learning Agent

**Definición (IBM):** https://www.ibm.com/think/topics/ai-agent-types

> Mejora su desempeño con el tiempo adaptándose a nueva experiencia y
> feedback, en vez de depender de reglas o modelos fijos. Se compone de:
> **elemento de desempeño**, **elemento de aprendizaje**, **crítico** y
> **generador de problemas**.

**Ejemplo de este notebook:** un agente de reposición de inventario que,
con el tiempo, va aprendiendo (vía un archivo JSON local que persiste
**entre ejecuciones**) qué estantes tienden a acertar o fallar sus
recomendaciones, y ajusta su confianza en consecuencia (refuerzo
simple).

**Requisitos:**
```bash
ollama pull llama3.2
pip install -r requirements.txt
```


In [ ]:
import os

from dotenv import load_dotenv

load_dotenv()

# Backend de modelo a usar. Puedes editar el default de abajo directamente
# (recomendado) o sobreescribirlo con la variable de entorno AGENT_MODEL
# antes de lanzar Jupyter.
AGENT_MODEL = os.environ.get("AGENT_MODEL", "llama3.2")


def _resolver_modelo(nombre: str):
    """Permite comparar el mismo agente con distintos backends de modelo
    sin tocar el resto del notebook -- solo cambia AGENT_MODEL arriba."""
    if nombre == "gemma-lmstudio":
        # LM Studio expone un servidor local compatible con la API de
        # OpenAI (pestaña "Local Server" dentro de LM Studio). Import
        # diferido: si no vas a usar este backend, no hace falta tener
        # langchain-openai instalado.
        #
        # Si corres esto en WSL y LM Studio está en Windows, "localhost"
        # puede no resolver hacia el host. Ajusta LMSTUDIO_BASE_URL con la
        # IP del host Windows vista desde WSL (ip route show | grep -i
        # default), ej: http://172.x.x.1:1234/v1
        from langchain_openai import ChatOpenAI

        base_url = os.environ.get("LMSTUDIO_BASE_URL", "http://172.30.32.1:1234/v1")
        return ChatOpenAI(
            model="google/gemma-4-e4b",
            base_url=base_url,
            api_key="lm-studio",  # LM Studio no valida la key, pero el cliente exige un valor no vacío
        )
    if nombre in ("llama3.2", "phi4-mini"):
        return f"ollama:{nombre}"
    raise ValueError(
        f"AGENT_MODEL desconocido: {nombre!r}. "
        "Opciones: llama3.2, phi4-mini, gemma-lmstudio"
    )


print(f"[CONFIG] Usando modelo: {AGENT_MODEL}")


## 5.1 Entorno simulado + memoria de aprendizaje persistente
A diferencia de los notebooks anteriores, aquí el 'conocimiento' del agente se guarda en disco y sobrevive a reinicios del notebook/kernel.

In [ ]:
import json
from pathlib import Path

from langchain.agents import create_agent
from langchain.tools import tool

SHELVES = {
    "E-12": {"producto": "auriculares bluetooth", "stock": 40},
    "E-27": {"producto": "cargador USB-C", "stock": 12},
    "E-33": {"producto": "mouse inalámbrico", "stock": 3},
    "E-41": {"producto": "teclado mecánico", "stock": 0},
}

LEARNING_STORE_PATH = Path("05_learning_store.json")


def _cargar_aprendizaje() -> dict:
    if LEARNING_STORE_PATH.exists():
        return json.loads(LEARNING_STORE_PATH.read_text())
    return {}


def _guardar_aprendizaje(data: dict) -> None:
    LEARNING_STORE_PATH.write_text(json.dumps(data, indent=2, ensure_ascii=False))


def _consultar_confianza(estante: str) -> dict:
    data = _cargar_aprendizaje()
    return data.get(estante, {"aciertos": 0, "fallos": 0, "confianza": 0.5,
                               "nota": "sin historial previo, usando confianza neutra"})


## 5.2 El "crítico" y el elemento de aprendizaje

`_registrar_resultado` juega el rol de **crítico** (evalúa si la acción
fue buena o mala) y de **elemento de aprendizaje** (ajusta el
conocimiento — aquí, un score de confianza — con ese feedback).

In [ ]:
def _registrar_resultado(estante: str, acierto: bool) -> dict:
    data = _cargar_aprendizaje()
    registro = data.setdefault(estante, {"aciertos": 0, "fallos": 0, "confianza": 0.5})
    if acierto:
        registro["aciertos"] += 1
    else:
        registro["fallos"] += 1
    total = registro["aciertos"] + registro["fallos"]
    registro["confianza"] = round(registro["aciertos"] / total, 2) if total else 0.5
    data[estante] = registro
    _guardar_aprendizaje(data)
    return registro


## 5.3 Tools: elemento de desempeño + generador de problemas + crítico

In [ ]:
@tool
def consultar_estado_estante(codigo_estante: str) -> str:
    """Consulta stock actual y CONFIANZA APRENDIDA (histórico de aciertos/
    fallos) para las recomendaciones de reposición de un estante."""
    print(f"[TOOL CALL] consultar_estado_estante(codigo_estante={codigo_estante!r})")
    info = SHELVES.get(codigo_estante)
    if info is None:
        resultado = f"Estante {codigo_estante} no existe."
    else:
        conf = _consultar_confianza(codigo_estante)
        resultado = (
            f"{codigo_estante} ({info['producto']}): stock={info['stock']}. "
            f"Historial aprendido -> aciertos={conf.get('aciertos', 0)}, "
            f"fallos={conf.get('fallos', 0)}, confianza={conf.get('confianza', 0.5)}"
        )
    print(f"[TOOL RESULT] consultar_estado_estante -> {resultado!r}")
    return resultado


@tool
def recomendar_cantidad_reposicion(codigo_estante: str, cantidad_sugerida: int) -> str:
    """Elemento de desempeño: dada una cantidad sugerida, la ajusta según
    la confianza aprendida (si la confianza es baja, sugiere ser más
    conservador y probar una cantidad menor -- esto es el 'generador de
    problemas' explorando una alternativa)."""
    print(
        f"[TOOL CALL] recomendar_cantidad_reposicion(codigo_estante={codigo_estante!r}, "
        f"cantidad_sugerida={cantidad_sugerida!r})"
    )
    conf = _consultar_confianza(codigo_estante).get("confianza", 0.5)
    if conf < 0.4:
        ajustada = max(1, cantidad_sugerida // 2)
        resultado = (f"Confianza baja ({conf}) para {codigo_estante}: en vez de "
                     f"{cantidad_sugerida}, se recomienda EXPLORAR con {ajustada} "
                     "unidades para reducir riesgo de sobre-stock.")
    else:
        resultado = f"Confianza aceptable ({conf}) para {codigo_estante}: se mantiene la cantidad sugerida ({cantidad_sugerida})."
    print(f"[TOOL RESULT] recomendar_cantidad_reposicion -> {resultado!r}")
    return resultado


@tool
def registrar_feedback_reposicion(codigo_estante: str, fue_correcta: bool) -> str:
    """Crítico + elemento de aprendizaje: registra si una reposición pasada
    fue correcta o no, y actualiza la confianza aprendida para ese estante
    de forma PERSISTENTE (sobrevive a futuras ejecuciones)."""
    print(
        f"[TOOL CALL] registrar_feedback_reposicion(codigo_estante={codigo_estante!r}, "
        f"fue_correcta={fue_correcta!r})"
    )
    if codigo_estante not in SHELVES:
        resultado = f"Estante {codigo_estante} no existe."
    else:
        registro = _registrar_resultado(codigo_estante, fue_correcta)
        resultado = f"Feedback registrado para {codigo_estante}. Nueva confianza aprendida: {registro['confianza']}"
    print(f"[TOOL RESULT] registrar_feedback_reposicion -> {resultado!r}")
    return resultado


## 5.4 Definición del agente que aprende

In [ ]:
SYSTEM_PROMPT = """
Eres un agente de reposición de inventario que APRENDE con el tiempo.
Antes de recomendar, consulta siempre el historial aprendido con
`consultar_estado_estante`. Usa `recomendar_cantidad_reposicion` para
ajustar tu sugerencia según la confianza histórica (no según reglas
fijas). Si el usuario te da feedback sobre una reposición pasada (si fue
correcta o no), regístralo con `registrar_feedback_reposicion` para que
el agente sea mejor la próxima vez que se le consulte por ese estante.
"""

agent = create_agent(
    model=_resolver_modelo(AGENT_MODEL),
    tools=[consultar_estado_estante, recomendar_cantidad_reposicion, registrar_feedback_reposicion],
    system_prompt=SYSTEM_PROMPT,
)


def _imprimir_secuencia_mensajes(mensajes: list) -> None:
    """Imprime, paso a paso, qué hizo el agente: si llamó a una tool (y con
    qué argumentos) o si solo produjo texto. Sirve para verificar -- sin
    depender de LangSmith -- qué tools se invocaron y en qué orden."""
    print("  --- secuencia de mensajes del agente ---")
    for i, msg in enumerate(mensajes):
        tipo = type(msg).__name__
        tool_calls = getattr(msg, "tool_calls", None)
        if tool_calls:
            for tc in tool_calls:
                print(f"    [{i}] {tipo} -> TOOL_CALL {tc['name']}(args={tc['args']})")
        elif tipo == "ToolMessage":
            print(f"    [{i}] {tipo} (resultado de {msg.name}): {msg.content!r}")
        else:
            contenido = getattr(msg, "content", "")
            print(f"    [{i}] {tipo}: {contenido!r}")
    print("  --- fin secuencia ---")


def consultar(instruccion: str) -> str:
    print(f"\n[AGENTE] Invocando episodio nuevo para: {instruccion!r}")
    resultado = agent.invoke({"messages": [{"role": "user", "content": instruccion}]})
    _imprimir_secuencia_mensajes(resultado["messages"])
    return resultado["messages"][-1].content


## 5.5 Ronda 1: sin historial (confianza neutra)

In [ ]:
print(consultar("¿Cuánto debería reponer del estante E-33? Sugiero 20 unidades."))


## 5.6 El usuario da feedback negativo -> el agente aprende

In [ ]:
print(consultar(
    "La última reposición de 20 unidades en E-33 fue un ERROR, generó "
    "sobre-stock. Registra ese feedback."
))


## 5.7 Ronda 2: la recomendación ya refleja lo aprendido

In [ ]:
print(consultar("¿Cuánto debería reponer ahora del estante E-33? Sugiero 20 unidades."))


## 5.8 Para reflexionar

- Vuelve a ejecutar este notebook desde cero (reinicia el kernel) SIN
  borrar `05_learning_store.json`: la confianza aprendida en la sesión
  anterior sigue ahí. Bórralo si quieres reiniciar el aprendizaje.
- Compara los 5 notebooks: cada uno añade una capacidad que el anterior
  no tenía (percepción → memoria → planificación → comparación de
  opciones → aprendizaje). Así es como IBM describe la evolución de
  complejidad entre los 5 tipos de agentes, y cómo, combinados, dan
  lugar a los **sistemas multiagente**.